In [1]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import umap

In [2]:
UMAP_PARAMS = dict(
    n_components=2,
    n_neighbors=50,
    min_dist=0.01,
    metric="cosine",
    verbose=True,
    n_jobs=16,
)

# Loading data

In [3]:
DATA_DIR = Path("../../data/playlist")
RANDOM_SEED = 0

EMB_PATH    = DATA_DIR / "embeddings_pure_bolt.parquet"
LOOKUP_PATH = DATA_DIR / "track_lookup.parquet"
VOCAB_PATH  = DATA_DIR / "training_vocab.parquet"

PLAYLIST_COUNT_FIT = 250   # tracks must appear in >= N playlists
ARTIST_CAP_FIT     = 10    # max tracks per artist in fit set

EMB_COLS = [f"e{i}" for i in range(128)]

In [4]:
def load_data():
    print("Loading embeddings …")
    embs = pd.read_parquet(EMB_PATH)                      # track_rowid + e0..e127

    print("Loading track lookup …")
    lookup = pd.read_parquet(
        LOOKUP_PATH,
        columns=["track_rowid", "track_name", "track_popularity", "artist_rowid", "artist_name", "album_name", "album_rowid"]
    )

    print("Loading training vocab (playlist_count) …")
    vocab = pd.read_parquet(
        VOCAB_PATH,
        columns=["track_rowid", "playlist_count"]
    )

    print("Joining …")
    df = lookup.merge(embs,  on="track_rowid", how="inner")
    df = df.merge(vocab, on="track_rowid", how="left")
    # tracks not in training vocab have playlist_count NaN → treat as 0
    df["playlist_count"] = df["playlist_count"].fillna(0).astype(int)

    print(f"  Total tracks after join: {len(df):,}")
    return df

df = load_data()

Loading embeddings …
Loading track lookup …
Loading training vocab (playlist_count) …
Joining …
  Total tracks after join: 5,412,049


# Landmarks albums and artists

In [5]:
# --- Compute entity mean embeddings ---
artist_embs = (
    df.groupby("artist_rowid")
      .agg(
          artist_name=("artist_name", "first"),
          max_popularity=("track_popularity", "max"),
          **{c: (c, "mean") for c in EMB_COLS},
      )
      .reset_index()
      .query("max_popularity >= 50")
)

album_embs = (
    df.groupby("album_rowid")
      .agg(
          album_name=("album_name", "first"),
          artist_name=("artist_name", "first"),
          max_popularity=("track_popularity", "max"),
          n_tracks=("track_rowid", "count"),
          **{c: (c, "mean") for c in EMB_COLS},
      )
      .reset_index()
      .query("max_popularity >= 50 & n_tracks >= 3")
)

print(f"Artists: {len(artist_embs):,}  Albums: {len(album_embs):,}")

Artists: 43,195  Albums: 53,433


In [6]:
# --- OndaRock notable entities via TF-IDF char n-gram cosine similarity ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

OR_PATH = Path("../../data/ondarock_milestones.csv")
or_df   = pd.read_csv(OR_PATH)

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))

# Artists
artist_name_list = artist_embs["artist_name"].str.lower().tolist()
or_artists       = or_df["artist"].str.lower().unique().tolist()

choice_mat = vec.fit_transform(artist_name_list)
query_mat  = vec.transform(or_artists)
sim        = cosine_similarity(query_mat, choice_mat)   # (n_queries, n_choices)

best_idxs   = sim.argmax(axis=1)
best_scores = sim[np.arange(len(or_artists)), best_idxs]

ARTIST_COSINE_THR = 0.6
NOTABLE_ARTISTS = {
    str(artist_embs.iloc[idx]["artist_rowid"])
    for score, idx in zip(best_scores, best_idxs)
    if score >= ARTIST_COSINE_THR
}
print(f"Matched {len(NOTABLE_ARTISTS)} notable artists (thr={ARTIST_COSINE_THR})")

# Albums — refit on "artist - album" combined keys
album_embs["_match_key"] = (
    album_embs["artist_name"].str.lower() + " - " +
    album_embs["album_name"].str.lower()
)
album_key_list = album_embs["_match_key"].tolist()
or_album_keys  = (or_df["artist"].str.lower() + " - " + or_df["album"].str.lower()).tolist()

choice_mat = vec.fit_transform(album_key_list)
query_mat  = vec.transform(or_album_keys)
sim        = cosine_similarity(query_mat, choice_mat)

best_idxs   = sim.argmax(axis=1)
best_scores = sim[np.arange(len(or_album_keys)), best_idxs]

ALBUM_COSINE_THR = 0.6
NOTABLE_ALBUMS = {
    str(album_embs.iloc[idx]["album_rowid"])
    for score, idx in zip(best_scores, best_idxs)
    if score >= ALBUM_COSINE_THR
}
print(f"Matched {len(NOTABLE_ALBUMS)} notable albums  (thr={ALBUM_COSINE_THR})")

Matched 629 notable artists (thr=0.6)
Matched 467 notable albums  (thr=0.6)


# Fitting UMAP

In [7]:
fit_set = (
    df[df["playlist_count"] >= PLAYLIST_COUNT_FIT]
    .sort_values("playlist_count", ascending=False)
    .groupby("artist_rowid", group_keys=False)
    .head(ARTIST_CAP_FIT)
    .reset_index(drop=True)
)
print(f"Fit set: {len(fit_set):,} tracks")

Fit set: 330,764 tracks


In [8]:
fit_matrix = fit_set[EMB_COLS].to_numpy(dtype=np.float32)
print(f"Fitting UMAP on {len(fit_matrix):,} tracks …")
reducer = umap.UMAP(**UMAP_PARAMS)
fit_coords = reducer.fit_transform(fit_matrix)

Fitting UMAP on 330,764 tracks …
UMAP(angular_rp_forest=True, metric='cosine', min_dist=0.01, n_jobs=16, n_neighbors=50, verbose=True)
Sat Mar  7 13:38:19 2026 Construct fuzzy simplicial set
Sat Mar  7 13:38:20 2026 Finding Nearest Neighbors
Sat Mar  7 13:38:20 2026 Building RP forest with 34 trees
Sat Mar  7 13:38:26 2026 NN descent for 18 iterations
	 1  /  18
	 2  /  18
	Stopping threshold met -- exiting after 2 iterations
Sat Mar  7 13:38:49 2026 Finished Nearest Neighbor Search
Sat Mar  7 13:38:54 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Sat Mar  7 13:41:45 2026 Finished embedding


In [9]:
print("Transforming artist embeddings …")
artist_coords = reducer.transform(artist_embs[EMB_COLS].to_numpy(dtype=np.float32))

print("Transforming album embeddings …")
album_coords  = reducer.transform(album_embs[EMB_COLS].to_numpy(dtype=np.float32))

def _coords_df(src, type_name, id_col, name_col, coords):
    return pd.DataFrame({
        "entity_type": type_name,
        "entity_id":   src[id_col].astype(str).values,
        "entity_name": src[name_col].values,
        "umap_x":      coords[:, 0].astype(np.float32),
        "umap_y":      coords[:, 1].astype(np.float32),
    })

tracks_c  = _coords_df(fit_set,     "track",  "track_rowid",  "track_name",  fit_coords)
artists_c = _coords_df(artist_embs, "artist", "artist_rowid", "artist_name", artist_coords)
albums_c  = _coords_df(album_embs,  "album",  "album_rowid",  "album_name",  album_coords)

combined_coords = pd.concat([tracks_c, artists_c, albums_c], ignore_index=True)
print(combined_coords.groupby("entity_type").size())

Transforming artist embeddings …
Sat Mar  7 13:41:45 2026 Building hub-based search tree
Sat Mar  7 13:41:52 2026 Forward diversification reduced edges from 16538200 to 2857052
Sat Mar  7 13:41:53 2026 Reverse diversification reduced edges from 2857052 to 2857052
Sat Mar  7 13:41:55 2026 Degree pruning reduced edges from 3462334 to 3462334
Sat Mar  7 13:41:55 2026 Resorting data and graph based on tree order
Sat Mar  7 13:41:55 2026 Building and compiling search function


Epochs completed:   0%|            0/30 [00:00]

	completed  0  /  30 epochs
	completed  3  /  30 epochs
	completed  6  /  30 epochs
	completed  9  /  30 epochs
	completed  12  /  30 epochs
	completed  15  /  30 epochs
	completed  18  /  30 epochs
	completed  21  /  30 epochs
	completed  24  /  30 epochs
	completed  27  /  30 epochs
Transforming album embeddings …


Epochs completed:   0%|            0/30 [00:00]

	completed  0  /  30 epochs
	completed  3  /  30 epochs
	completed  6  /  30 epochs
	completed  9  /  30 epochs
	completed  12  /  30 epochs
	completed  15  /  30 epochs
	completed  18  /  30 epochs
	completed  21  /  30 epochs
	completed  24  /  30 epochs
	completed  27  /  30 epochs
entity_type
album      53433
artist     43195
track     330764
dtype: int64


# Plotting

In [10]:
# --- Multi-entity UMAP: Cell 5 — Visualize tracks + artists + albums + labels ---
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.style.use("dark_background")

tc  = combined_coords[combined_coords.entity_type == "track"]
ac  = combined_coords[combined_coords.entity_type == "artist"]
alc = combined_coords[combined_coords.entity_type == "album"]

fig, ax = plt.subplots(figsize=(75, 75))

# Layer 1 — track density
_, _, _, img = ax.hist2d(tc.umap_x, tc.umap_y, bins=2000, cmap="inferno",
                         norm=mcolors.PowerNorm(gamma=0.3))

# Layer 2 — notable albums (deepskyblue dot + small label)
sel_al = alc[alc.entity_id.isin(NOTABLE_ALBUMS)]
ax.scatter(sel_al.umap_x, sel_al.umap_y, s=10, color="deepskyblue", zorder=6, linewidths=0)
for _, row in sel_al.iterrows():
    ax.annotate(
        row.entity_name,
        (row.umap_x, row.umap_y),
        fontsize=2, color="deepskyblue",
        xytext=(4, 4), textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.4, linewidth=0),
    )

# Layer 3 — labeled artists (white dot + bold label)
sel_a = ac[ac.entity_id.isin(NOTABLE_ARTISTS)]
ax.scatter(sel_a.umap_x, sel_a.umap_y, s=5, color="white", zorder=5, linewidths=0)
for _, row in sel_a.iterrows():
    ax.annotate(
        row.entity_name,
        (row.umap_x, row.umap_y),
        fontsize=4, fontweight="bold", color="white",
        xytext=(4, 4), textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5, linewidth=0),
    )

ax.set_axis_off()
fig.tight_layout()

p = UMAP_PARAMS
plot_name = (
    f"umap_nn{p['n_neighbors']}_md{p['min_dist']}"
    f"_{p['metric']}_pc{PLAYLIST_COUNT_FIT}"
).replace(".", "d")
fig.savefig(f"plots/{plot_name}.png", dpi=300, bbox_inches="tight")
print(f"Saved → {plot_name}")
plt.show()

Saved → umap_nn50_md0d01_cosine_pc250
